In [10]:
from dotenv import load_dotenv
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
import os
from pydantic import BaseModel,Field
from agents import Agent,WebSearchTool,trace,Runner,function_tool
from agents.model_settings import ModelSettings
import asyncio

In [11]:
load_dotenv(override=True)

True

In [12]:
INSTRUCTION=f"""
            You are able the send the email with provided subject and body
            """

@function_tool
def send_email(subject:str,body:str)->dict[str,str]:
    "Send email with the given subject and HTML body"
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("nguyenhoangdangquan2804@gmail.com")  # Change to your verified sender
    to_email = To("nguyenhoangdangquan2804@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"response":"sucessful","message":"The email was already sent"}

SendEmailAgent=Agent(name="SendEmailAgent",
                            model="gpt-4o-mini",
                            instructions=INSTRUCTION,
                            tools=[send_email],
                            model_settings=ModelSettings(tool_choice="required"))

In [13]:
HOW_MANY_SEARCHES=3

INSTRUCTION=f"""You are an expert research strategist and data analyst. Your objective is to deconstruct the user's question and formulate a highly targeted search plan.
Generate exactly {HOW_MANY_SEARCHES} distinct, keyword-optimized web search queries designed to uncover authoritative, factual information. Use advanced search operators (like site: or exact match quotes) where appropriate.
CRITICAL: Do not attempt to answer the user's question. Your only job is to populate the required response schema with your search queries
            """
class WebSearchItem(BaseModel):
    reason:str=Field("The reason why it is relevant to the question")
    query:str=Field("The search term to find the website")

class WebSearchTerms(BaseModel):
    list_terms:list[WebSearchItem]=Field("List of terms that are best relevant to the question")

SearchingTermAgent=Agent(name="SearchingTermAgent",
                                model="gpt-4o-mini",
                                instructions=INSTRUCTION,
                                output_type=WebSearchTerms)

In [14]:
INSTRUCTION="""You are a senior investigative journalist. Your objective is to research the provided term and distill the findings into a concise executive briefing.
INSTRUCTIONS:
Write a comprehensive summary strictly under 300 words.
Focus entirely on the main points, eliminating all fluff, filler, and secondary information.
Output the final summary directly. Do not include any conversational text (e.g., do not say 'Here is the summary:')
            """
            
SearchingAgent=Agent(name="SearchingAgent",
                        model="gpt-4o-mini",
                        instructions=INSTRUCTION,
                        tools=[WebSearchTool(search_context_size='low')],
                        model_settings=ModelSettings(tool_choice="required"))

In [15]:
INSTRUCTION="""You are a senior investigative journalist and expert data synthesizer. Your objective is to distill the provided text into a highly concentrated executive summary.
CRITICAL INSTRUCTIONS:
Length Limit: Strictly under 300 words.
Content: Extract only the core facts, primary arguments, and critical data points. Ruthlessly eliminate all fluff, secondary context, and filler.
Output Format: Output ONLY the raw summary text. Absolutely no conversational preamble, pleasantries, or concluding remarks (e.g., do not say 'Here is the summary:').
            """
            
class ReportSummary(BaseModel):
    title:str=Field("A short 2-3 sentense of the summary")
    body:str=Field("Summary report")

SummaryAgent=Agent(name="SummaryAgent",
                        model="gpt-4o-mini",
                        instructions=INSTRUCTION,
                        output_type=ReportSummary)

In [16]:
async def searching_term(question:str):
    print("Start searching")
    response=await Runner.run(SearchingTermAgent,question)
    print(f"Finding results, {len(response.final_output.list_terms)} searches")
    return response.final_output

async def searching(WebSearchTerms:WebSearchTerms):
    print("Start searching detail")
    tasks=[asyncio.create_task(Runner.run(SearchingAgent,item.query)) for item in WebSearchTerms.list_terms]
    response=await asyncio.gather(*tasks)
    final_outputs=[item.final_output for item in response]
    print("Finish searching")
    return final_outputs

async def summary_report(question,searching_result:list[str]):
    print("Start summarizing")
    formatted_results = '\n\n---\n\n'.join(searching_result)
    input = f"Original query: {question}\nSummarized search results: {formatted_results}"
    response=await Runner.run(SummaryAgent,input)
    print("Finish summarizing")
    return response.final_output

async def send_email(ReportSummary:ReportSummary):
    print("Start sending email")
    input=f""""Subject: {ReportSummary.title}, Email body: {ReportSummary.body}"""
    await Runner.run(SendEmailAgent,input)
    print("Email was sent")
    return None

In [23]:
question="In Vietnam stock market, which stock is valuable?"

In [24]:
with trace("Finding news"):
    SearchingTerms=await searching_term(question)
    SearchingDetail=await searching(SearchingTerms)
    SummaryReport=await summary_report(question=question,searching_result=SearchingDetail)
    await send_email(SummaryReport)

Start searching
Finding results, 3 searches
Start searching detail
Finish searching
Start summarizing
Finish summarizing
Start sending email
Email was sent
